# 14_05_OpenTelemetry_で_RAG＋エージェ

コード：`ch14/blocks/14_05_OpenTelemetry_で_RAG＋エージェ.ipynb`

本文の断片は参照表示し、末尾のセルで前提を補う既存ランナーを実行します。関数・変数・外部処理が省略されているため、断片だけを単独実行しないでください。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import os
import pathlib
import sys
import zipfile

CODE = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents,
                        pathlib.Path.cwd() / 'gradient-attention-agent']
             if (p / 'run_all.py').is_file() and (p / 'common/blocks.py').is_file()), None)
if CODE is None:
    from google.colab import files
    uploaded = files.upload()  # gradient-attention-agent-code.zip を選ぶ
    if len(uploaded) != 1:
        raise ValueError('配布ZIPを1つ選んでください')
    dest = pathlib.Path.cwd().resolve()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        for member in z.namelist():
            if not (dest / member).resolve().is_relative_to(dest / 'gradient-attention-agent'):
                raise ValueError('同梱のコード配布ZIPを選んでください')
        z.extractall(dest)
    CODE = dest / 'gradient-attention-agent'
    del uploaded
print('コード一式:', CODE)


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


## 本文の掲載コード（参照）


```python
def run_agent(task, max_steps=8):
    with tracer.start_as_current_span("agent.run") as root:
        root.set_attribute("agent.max_steps", max_steps)
        # ★ trace ID を応答に返す。ユーザー報告からトレースへ一発で飛べる
        trace_id = format(root.get_span_context().trace_id, "032x")
        messages, used = [{"role": "user", "content": task}], []

        for step in range(max_steps):
            with tracer.start_as_current_span(f"agent.step.{step}") as sp:
                r = call_llm(messages)
                calls = extract_tool_calls(r)
                sp.set_attribute("agent.tool_calls", len(calls))
                if not calls:
                    sp.set_attribute("agent.stop_reason", "final_answer")
                    return {"text": final_text(r), "trace_id": trace_id}

                for c in calls:
                    with tracer.start_as_current_span("execute_tool") as ts:
                        ts.set_attribute("gen_ai.operation.name", "execute_tool")
                        ts.set_attribute("gen_ai.tool.name", c.name)
                        ts.set_attribute("gen_ai.tool.arguments.sha256", digest(str(c.args)))
                        ok, out = execute(c)
                        ts.set_attribute("tool.ok", ok)
                        if not ok:
                            ts.set_attribute("error.type", out.split(":")[0][:64])  # 種類だけ
                            # ★ 状態を ERROR にしないと、tail sampling の「失敗は全部残す」に掛からない
                            ts.set_status(Status(StatusCode.ERROR, out[:200]))
                    used.append(c.name)
                messages += to_messages(r, calls)

        root.set_attribute("agent.stop_reason", "max_steps")   # ★ 打ち切りも記録
        root.set_attribute("agent.tools_used", used)
        return {"text": None, "trace_id": trace_id}
```


## 前提を補って実行する

既存ランナーが関連する断片と必要なデータ・スタブを接続します。外部サービスを実行した証明や、掲載用AI応答の完全証跡にはなりません。


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(CODE / "run_all.py"), "--only", "14", "--name=otel", "--skip-docker"], cwd=CODE, check=True)
